# 10 — R-Drop + Label Smoothing (Phase 2)

**Goal:** Apply two advanced regularization techniques to BanglaBERT and measure improvement over the plain baseline.

**Techniques:**
1. **Label Smoothing (LS)** — Soft targets (e.g., 0.9 instead of 1.0) prevent overconfidence
2. **R-Drop** — Two forward passes with different dropout, KL-divergence penalty forces consistency
3. **R-Drop + LS** — Combined

**Backbone:** BanglaBERT (csebuetnlp/banglabert) — Phase 1 winner  
**Datasets:** All 3 binary + BanglaSarc3 ternary = 4 datasets × 3 techniques = **12 runs**  
**Baselines:** Plain BanglaBERT from nb05 (binary) and nb08 (ternary)

In [1]:
!pip install --upgrade transformers huggingface_hub -q


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, sys, json, time, warnings, gc, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')

# ── Paths ──
if os.path.exists('/kaggle/working'):
    REPO_ROOT = Path('/kaggle/working/Sarcasm_detection')
    BIG_TMP = Path('/kaggle/tmp')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '00_admin').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    BIG_TMP = REPO_ROOT / '_tmp'

PROJECT    = REPO_ROOT
SPLIT_DIR  = PROJECT / '01_data' / 'interim' / 'splits'
TABLE_DIR  = PROJECT / '04_outputs' / 'tables'
PRED_DIR   = PROJECT / '04_outputs' / 'predictions'
LOG_DIR    = PROJECT / '04_outputs' / 'run_logs'
REPORT_DIR = PROJECT / '04_outputs' / 'reports'

for d in [TABLE_DIR, PRED_DIR, LOG_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR   = BIG_TMP / 'hf_cache'
TEMP_TRAIN_DIR = BIG_TMP / 'train_cache'
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEMP_TRAIN_DIR.mkdir(parents=True, exist_ok=True)

def disk_free_gb(path='/'):
    try: return shutil.disk_usage(path).free / 1e9
    except: return shutil.disk_usage('.').free / 1e9

def clear_hf_cache():
    if HF_CACHE_DIR.exists():
        shutil.rmtree(HF_CACHE_DIR, ignore_errors=True)
        HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project: {PROJECT}")
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  {props.name} — {props.total_memory / 1e9:.1f} GB")
if os.path.exists('/kaggle/working'):
    print(f"Disk (working): {disk_free_gb('/kaggle/working'):.1f} GB")
    print(f"Disk (tmp): {disk_free_gb(BIG_TMP):.1f} GB")
    clear_hf_cache()

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project: /Users/sefayet/Desktop/Github/Sarcasm_detection
GPU: False


In [3]:
# ── Configuration ──

MODEL_NAME = 'csebuetnlp/banglabert'
MODEL_TAG  = 'banglabert'

DATASETS = {
    'ben_sarc_binary': {
        'train': SPLIT_DIR / 'ben_sarc_binary_train.csv',
        'val':   SPLIT_DIR / 'ben_sarc_binary_val.csv',
        'test':  SPLIT_DIR / 'ben_sarc_binary_test.csv',
        'num_labels': 2, 'label_col': 'label_binary',
    },
    'banglasarc_binary': {
        'train': SPLIT_DIR / 'banglasarc_binary_train.csv',
        'val':   SPLIT_DIR / 'banglasarc_binary_val.csv',
        'test':  SPLIT_DIR / 'banglasarc_binary_test.csv',
        'num_labels': 2, 'label_col': 'label_binary',
    },
    'banglasarc3_binary': {
        'train': SPLIT_DIR / 'banglasarc3_binary_train.csv',
        'val':   SPLIT_DIR / 'banglasarc3_binary_val.csv',
        'test':  SPLIT_DIR / 'banglasarc3_binary_test.csv',
        'num_labels': 2, 'label_col': 'label_binary',
    },
    'banglasarc3_ternary': {
        'train': SPLIT_DIR / 'banglasarc3_ternary_train.csv',
        'val':   SPLIT_DIR / 'banglasarc3_ternary_val.csv',
        'test':  SPLIT_DIR / 'banglasarc3_ternary_test.csv',
        'num_labels': 3, 'label_col': 'label_original',
    },
}

# Ternary label mapping
LABEL_MAP = {'Non-Sarcastic': 0, 'Neutral': 1, 'Sarcastic': 2}

# Techniques to test
TECHNIQUES = {
    'label_smoothing':      {'ls': 0.1, 'rdrop_alpha': 0.0},
    'rdrop':                {'ls': 0.0, 'rdrop_alpha': 5.0},
    'rdrop_plus_ls':        {'ls': 0.1, 'rdrop_alpha': 5.0},
}

MAX_LENGTH    = 128
EPOCHS        = 3
BATCH_SIZE    = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
SEED          = 42
TEXT_COL      = 'text'

total_runs = len(TECHNIQUES) * len(DATASETS)
print(f"Techniques: {list(TECHNIQUES.keys())}")
print(f"Datasets: {list(DATASETS.keys())}")
print(f"Total runs: {total_runs}")

Techniques: ['label_smoothing', 'rdrop', 'rdrop_plus_ls']
Datasets: ['ben_sarc_binary', 'banglasarc_binary', 'banglasarc3_binary', 'banglasarc3_ternary']
Total runs: 12


In [4]:
# ── Dataset class ──

class SarcasmDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding='max_length',
                                   max_length=max_length, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


def load_split(csv_path, label_col):
    df = pd.read_csv(csv_path)
    texts = df[TEXT_COL].astype(str).tolist()
    if df[label_col].dtype == object:
        labels = df[label_col].map(LABEL_MAP).astype(int).tolist()
    else:
        labels = df[label_col].astype(int).tolist()
    return texts, labels


def compute_metrics_binary(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':    accuracy_score(labels, preds),
        'macro_f1':    f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
        'binary_f1':   f1_score(labels, preds, average='binary', pos_label=1),
        'precision':   precision_score(labels, preds, average='binary', pos_label=1),
        'recall':      recall_score(labels, preds, average='binary', pos_label=1),
    }


def compute_metrics_ternary(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':    accuracy_score(labels, preds),
        'macro_f1':    f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
        'macro_precision': precision_score(labels, preds, average='macro'),
        'macro_recall':    recall_score(labels, preds, average='macro'),
    }


# Sanity check
for ds_name, ds_cfg in DATASETS.items():
    tr, trl = load_split(ds_cfg['train'], ds_cfg['label_col'])
    print(f"{ds_name}: train={len(tr)}, labels={sorted(set(trl))}, num_labels={ds_cfg['num_labels']}")

ben_sarc_binary: train=20508, labels=[0, 1], num_labels=2
banglasarc_binary: train=4089, labels=[0, 1], num_labels=2
banglasarc3_binary: train=6413, labels=[0, 1], num_labels=2
banglasarc3_ternary: train=9657, labels=[0, 1, 2], num_labels=3


In [5]:
# ── R-Drop + Label Smoothing Trainer ──

class RDropTrainer(Trainer):
    """
    Custom Trainer supporting:
    - Label Smoothing (ls > 0)
    - R-Drop (rdrop_alpha > 0): two forward passes, KL-div penalty
    - Both combined
    """
    def __init__(self, *args, rdrop_alpha=0.0, ls=0.0, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.rdrop_alpha = rdrop_alpha
        self.ls = ls
        self.class_weights = class_weights  # Optional tensor for ternary

    def _ce_loss(self, logits, labels):
        """Cross-entropy with optional label smoothing and class weights."""
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        if self.ls > 0:
            return F.cross_entropy(logits, labels, weight=weight, label_smoothing=self.ls)
        else:
            return F.cross_entropy(logits, labels, weight=weight)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')

        if self.rdrop_alpha > 0:
            # Two forward passes with different dropout masks
            outputs1 = model(**inputs)
            outputs2 = model(**inputs)
            logits1 = outputs1.logits
            logits2 = outputs2.logits

            # CE loss (average of both passes)
            ce_loss = (self._ce_loss(logits1, labels) + self._ce_loss(logits2, labels)) / 2

            # KL divergence (symmetric)
            p = F.log_softmax(logits1, dim=-1)
            q = F.log_softmax(logits2, dim=-1)
            kl_loss = (
                F.kl_div(p, q.exp(), reduction='batchmean', log_target=False) +
                F.kl_div(q, p.exp(), reduction='batchmean', log_target=False)
            ) / 2

            loss = ce_loss + self.rdrop_alpha * kl_loss
            outputs = outputs1  # Use first pass outputs for predictions
        else:
            # Standard forward pass (label smoothing only)
            outputs = model(**inputs)
            loss = self._ce_loss(outputs.logits, labels)

        return (loss, outputs) if return_outputs else loss

In [6]:
# ── Training function ──

def train_advanced(technique_tag, technique_cfg, dataset_tag, dataset_cfg):
    run_label = f"{technique_tag} x {dataset_tag}"
    print(f"\n{'='*70}")
    print(f"  {run_label}")
    print(f"  LS={technique_cfg['ls']} | R-Drop alpha={technique_cfg['rdrop_alpha']}")
    if os.path.exists('/kaggle/working'):
        print(f"  Disk (tmp): {disk_free_gb(BIG_TMP):.1f} GB | working: {disk_free_gb('/kaggle/working'):.1f} GB")
    print(f"{'='*70}")
    t_start = time.time()

    num_labels = dataset_cfg['num_labels']
    label_col = dataset_cfg['label_col']
    is_ternary = num_labels == 3

    # Load data
    tr_texts, tr_labels = load_split(dataset_cfg['train'], label_col)
    vl_texts, vl_labels = load_split(dataset_cfg['val'], label_col)
    te_texts, te_labels = load_split(dataset_cfg['test'], label_col)
    print(f"  Train: {len(tr_texts)} | Val: {len(vl_texts)} | Test: {len(te_texts)}")

    # Tokenizer & model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=str(HF_CACHE_DIR))
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels, ignore_mismatched_sizes=True,
        cache_dir=str(HF_CACHE_DIR)
    )

    # Datasets
    train_ds = SarcasmDataset(tr_texts, tr_labels, tokenizer, MAX_LENGTH)
    val_ds   = SarcasmDataset(vl_texts, vl_labels, tokenizer, MAX_LENGTH)
    test_ds  = SarcasmDataset(te_texts, te_labels, tokenizer, MAX_LENGTH)

    # Class weights for ternary
    class_weights = None
    if is_ternary:
        counts = Counter(tr_labels)
        total = sum(counts.values())
        class_weights = torch.tensor(
            [total / (num_labels * counts[i]) for i in range(num_labels)],
            dtype=torch.float32
        )

    # Temp dir
    run_tmp = TEMP_TRAIN_DIR / f"{technique_tag}_{dataset_tag}"
    if run_tmp.exists(): shutil.rmtree(run_tmp)
    run_tmp.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(run_tmp),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        report_to='none',
        dataloader_num_workers=2,
    )

    metrics_fn = compute_metrics_ternary if is_ternary else compute_metrics_binary

    trainer = RDropTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=metrics_fn,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        rdrop_alpha=technique_cfg['rdrop_alpha'],
        ls=technique_cfg['ls'],
        class_weights=class_weights,
    )

    trainer.train()

    # Predict val
    val_out = trainer.predict(val_ds)
    val_preds = np.argmax(val_out.predictions, axis=-1)
    val_probs = torch.softmax(torch.tensor(val_out.predictions), dim=-1).numpy()
    print(f"  Val  — Acc: {val_out.metrics['test_accuracy']:.4f} | Macro-F1: {val_out.metrics['test_macro_f1']:.4f}")

    # Predict test
    test_out = trainer.predict(test_ds)
    test_preds = np.argmax(test_out.predictions, axis=-1)
    test_probs = torch.softmax(torch.tensor(test_out.predictions), dim=-1).numpy()
    print(f"  Test — Acc: {test_out.metrics['test_accuracy']:.4f} | Macro-F1: {test_out.metrics['test_macro_f1']:.4f}")

    # Save predictions
    for split_name, texts, labels, preds, probs in [
        ('test', te_texts, te_labels, test_preds, test_probs),
        ('val',  vl_texts, vl_labels, val_preds,  val_probs),
    ]:
        pred_dict = {'text': texts, 'true_label': labels, 'pred_label': preds}
        for c in range(num_labels):
            pred_dict[f'prob_{c}'] = probs[:, c]
        pd.DataFrame(pred_dict).to_csv(
            PRED_DIR / f"10_{technique_tag}_{dataset_tag}_{split_name}_predictions.csv",
            index=False
        )

    cm = confusion_matrix(te_labels, test_preds).tolist()
    t_elapsed = time.time() - t_start

    result = {
        'technique': technique_tag,
        'model_tag': MODEL_TAG,
        'dataset': dataset_tag,
        'num_labels': num_labels,
        'ls': technique_cfg['ls'],
        'rdrop_alpha': technique_cfg['rdrop_alpha'],
        'val_accuracy':  val_out.metrics['test_accuracy'],
        'val_macro_f1':  val_out.metrics['test_macro_f1'],
        'test_accuracy':  test_out.metrics['test_accuracy'],
        'test_macro_f1':  test_out.metrics['test_macro_f1'],
        'test_weighted_f1': test_out.metrics['test_weighted_f1'],
        'confusion_matrix': json.dumps(cm),
        'train_samples': len(tr_texts),
        'time_seconds': round(t_elapsed, 1),
    }

    # Cleanup
    del model, trainer, train_ds, val_ds, test_ds
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    if run_tmp.exists(): shutil.rmtree(run_tmp)
    clear_hf_cache()
    print(f"  Done in {t_elapsed/60:.1f} min")

    return result

In [7]:
# ── Run all experiments (with resume) ──

all_results = []
summary_path = TABLE_DIR / '10_rdrop_ls_summary.csv'

if summary_path.exists():
    prev_df = pd.read_csv(summary_path)
    all_results = prev_df.to_dict('records')
    done_keys = set(zip(prev_df['technique'], prev_df['dataset']))
    print(f"Resuming: {len(done_keys)} runs done")
else:
    done_keys = set()

if TEMP_TRAIN_DIR.exists():
    shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)
    TEMP_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
clear_hf_cache()

run_num = len(done_keys)

for tech_tag, tech_cfg in TECHNIQUES.items():
    for ds_tag, ds_cfg in DATASETS.items():
        if (tech_tag, ds_tag) in done_keys:
            print(f"Skipping {tech_tag} x {ds_tag} (done)")
            continue

        run_num += 1
        print(f"\n>>> Run {run_num}/{total_runs}")

        if os.path.exists('/kaggle/working') and disk_free_gb('/kaggle/working') < 3.0:
            clear_hf_cache()
            if disk_free_gb('/kaggle/working') < 2.0:
                raise RuntimeError(f"Disk too low: {disk_free_gb('/kaggle/working'):.1f} GB")

        try:
            result = train_advanced(tech_tag, tech_cfg, ds_tag, ds_cfg)
            all_results.append(result)
            pd.DataFrame(all_results).to_csv(summary_path, index=False)
            print(f"  Saved. {total_runs - run_num} remaining.")
        except Exception as e:
            print(f"  FAILED: {e}")
            if all_results:
                pd.DataFrame(all_results).to_csv(summary_path, index=False)
            if TEMP_TRAIN_DIR.exists():
                shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)
            clear_hf_cache()
            raise

if TEMP_TRAIN_DIR.exists():
    shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)
clear_hf_cache()

print(f"\n{'='*70}")
print(f"All {total_runs} runs complete!")


>>> Run 1/12

  label_smoothing x ben_sarc_binary
  LS=0.1 | R-Drop alpha=0.0
  Train: 20508 | Val: 2564 | Test: 2564


Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
# ── Results ──

results_df = pd.read_csv(summary_path)

print("="*90)
print("  R-DROP + LABEL SMOOTHING RESULTS (Test Macro-F1)")
print("="*90)
print(results_df[['technique', 'dataset', 'test_accuracy', 'test_macro_f1', 'test_weighted_f1', 'time_seconds']]
      .to_string(index=False, float_format='%.4f'))

In [ ]:
# ── Pivot: technique x dataset ──

pivot = results_df.pivot_table(
    index='technique', columns='dataset',
    values='test_macro_f1', aggfunc='first'
)

# Add baselines from previous notebooks
# Binary baselines from nb05
bb_path = TABLE_DIR / '05_baseline_banglabert_binary_summary_all_datasets.csv'
if bb_path.exists():
    bb_df = pd.read_csv(bb_path)
    bb_row = {}
    for _, r in bb_df.iterrows():
        ds = r.get('dataset') or r.get('dataset_tag') or r.get('dataset_name')
        f1 = r.get('test_macro_f1') or r.get('macro_f1')
        if ds and f1: bb_row[ds] = f1
    if bb_row: pivot.loc['plain_baseline (nb05)'] = bb_row

# FGM from nb06
fgm_path = TABLE_DIR / '06_fgm_banglabert_binary_summary_all_datasets.csv'
if fgm_path.exists():
    fgm_df = pd.read_csv(fgm_path)
    fgm_row = {}
    for _, r in fgm_df.iterrows():
        ds = r.get('dataset') or r.get('dataset_tag') or r.get('dataset_name')
        f1 = r.get('test_macro_f1') or r.get('macro_f1')
        if ds and f1: fgm_row[ds] = f1
    if fgm_row: pivot.loc['fgm_baseline (nb06)'] = fgm_row

# Ternary baseline from nb08
tern_path = TABLE_DIR / '08_multi_backbone_ternary_summary.csv'
if tern_path.exists():
    tern_df = pd.read_csv(tern_path)
    bb_tern = tern_df[tern_df['model_tag'] == 'banglabert']
    if len(bb_tern) > 0:
        pivot.loc['plain_baseline (nb05)']['banglasarc3_ternary'] = bb_tern.iloc[0]['test_macro_f1']

pivot['mean'] = pivot.mean(axis=1)
pivot = pivot.sort_values('mean', ascending=False)

print("="*90)
print("  MACRO-F1 COMPARISON — Techniques vs Baselines")
print("="*90)
print(pivot.to_string(float_format='%.4f'))
print()
print(f"Best technique: {pivot['mean'].idxmax()} (mean={pivot['mean'].max():.4f})")

In [ ]:
# ── Improvement over plain baseline ──

print("="*90)
print("  IMPROVEMENT OVER PLAIN BANGLABERT BASELINE")
print("="*90)

if 'plain_baseline (nb05)' in pivot.index:
    baseline_row = pivot.loc['plain_baseline (nb05)']
    for tech in TECHNIQUES.keys():
        if tech in pivot.index:
            deltas = pivot.loc[tech] - baseline_row
            print(f"\n{tech}:")
            for col in [c for c in deltas.index if c != 'mean']:
                d = deltas[col]
                if pd.notna(d):
                    sign = '+' if d >= 0 else ''
                    print(f"  {col}: {sign}{d:.4f}")
            print(f"  mean: {'+' if deltas['mean'] >= 0 else ''}{deltas['mean']:.4f}")

In [ ]:
# ── Save artifacts ──

cm_dict = {}
for _, row in results_df.iterrows():
    cm_dict[f"{row['technique']}_{row['dataset']}"] = json.loads(row['confusion_matrix'])
with open(TABLE_DIR / '10_rdrop_ls_confusion_matrices.json', 'w') as f:
    json.dump(cm_dict, f, indent=2)

results_df[['technique', 'dataset', 'num_labels', 'ls', 'rdrop_alpha',
            'train_samples', 'time_seconds']].to_csv(
    LOG_DIR / '10_rdrop_ls_run_metadata.csv', index=False
)

pivot.to_csv(TABLE_DIR / '10_rdrop_ls_macro_f1_pivot.csv')

print("All artifacts saved.")
print("\nFiles produced:")
for p in sorted(PROJECT.rglob('10_*')):
    if p.is_file():
        print(f"  {p.relative_to(PROJECT)}  ({p.stat().st_size / 1e3:.1f} KB)")